# Phase 5 : Modèles Supervisés — Classification

Ce notebook se concentre sur la Phase 5 du projet d'ingénierie ML. Nous transformons ici notre problématique de régression en une problématique de **Classification (Segmentation)**. 

L'objectif est de classer les transactions en trois segments de valeur : **Faible**, **Moyen**, et **Elevé**.

## 1. Importations et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import xgboost as xgb

# Style
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

import warnings
warnings.filterwarnings('ignore')

## 2. Chargement et Ingénierie du Target (Discrétisation)

Nous allons utiliser les percentiles (33% et 66%) de `log_Montant_TTC` pour créer nos trois classes.

In [ ]:
# Chargement du dataset
df = pd.read_csv('../data/processed/dataset_ml_features.csv')

# Calcul des seuils
q1 = df['log_Montant_TTC'].quantile(0.33)
q2 = df['log_Montant_TTC'].quantile(0.66)

def discretize_revenue(val):
    if val < q1:
        return 'Faible'
    elif val < q2:
        return 'Moyen'
    else:
        return 'Elevé'

# Création de la nouvelle variable cible
df['Segment_Valeur'] = df['log_Montant_TTC'].apply(discretize_revenue)

print("Seuils utilisés :")
print(f"- Faible if < {q1:.2f}")
print(f"- Moyen if between {q1:.2f} and {q2:.2f}")
print(f"- Elevé if > {q2:.2f}")

print("\nRépartition des classes :")
print(df['Segment_Valeur'].value_counts(normalize=True))

## 3. Préparation des Données

In [ ]:
# Définition de X et y
target_name = 'Segment_Valeur'
cols_to_drop = ['FK_Date', 'Montant_TTC', 'Montant_HT', 'log_Montant_TTC', target_name]

X = df.drop(columns=cols_to_drop)
y = df[target_name]

# Split stratifié pour conserver la distribution des classes
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## 4. Implémentation des Modèles

### 4.1 Régression Logistique (Multiclasse)

In [ ]:
log_reg = LogisticRegression(multi_class='ovr', max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)

print("Accuracy Logistic Regression:", accuracy_score(y_test, y_pred_log))

### 4.2 Random Forest Classifier

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

print("Accuracy Random Forest:", accuracy_score(y_test, y_pred_rf))

### 4.3 Support Vector Machine (SVC)

In [ ]:
svm_clf = SVC(kernel='rbf', probability=True)
svm_clf.fit(X_train, y_train)
y_pred_svm = svm_clf.predict(X_test)

print("Accuracy SVM:", accuracy_score(y_test, y_pred_svm))

### 4.4 XGBoost Classifier

In [ ]:
# XGBoost nécessite des labels numériques pour la classification
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_clf.fit(X_train, y_train_encoded)
y_pred_xgb_enc = xgb_clf.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

print("Accuracy XGBoost:", accuracy_score(y_test, y_pred_xgb))

## 5. Comparaison et Évaluation Détaillée

In [ ]:
models = {
    'Logistic Regression': y_pred_log,
    'Random Forest': y_pred_rf,
    'SVM': y_pred_svm,
    'XGBoost': y_pred_xgb
}

comparison = []
for name, pred in models.items():
    comparison.append({
        'Modèle': name,
        'Accuracy': accuracy_score(y_test, pred),
        'F1-Micro': f1_score(y_test, pred, average='micro'),
        'F1-Macro': f1_score(y_test, pred, average='macro')
    })

df_comp = pd.DataFrame(comparison).sort_values(by='Accuracy', ascending=False)
df_comp

### 5.1 Matrice de Confusion du Meilleur Modèle

In [ ]:
best_model_name = df_comp.iloc[0]['Modèle']
best_preds = models[best_model_name]

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, best_preds, labels=le.classes_ if best_model_name == 'XGBoost' else sorted(y_test.unique()))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_ if best_model_name == 'XGBoost' else sorted(y_test.unique()), 
            yticklabels=le.classes_ if best_model_name == 'XGBoost' else sorted(y_test.unique()))
plt.title(f"Matrice de Confusion : {best_model_name}")
plt.xlabel("Prédiction")
plt.ylabel("Réalité")
plt.show()

print(f"\nClassification Report for {best_model_name}:\n")
print(classification_report(y_test, best_preds))

## 6. Optimisation des Hyperparamètres

Nous optimisons le Random Forest qui offre souvent un bon équilibre.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'criterion': ['gini', 'entropy']
}

cv = StratifiedKFold(n_splits=3)
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=cv, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)

print("Best Params:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

## 7. Export du Modèle Final

In [ ]:
best_classifier = grid_search.best_estimator_

# Sauvegarde
model_path = '../models/classification_segment_model.joblib'
joblib.dump(best_classifier, model_path)

print(f"Modèle de classification sauvegardé dans : {model_path}")